# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing the **FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities in this notebook are referenced by their `@id` per the Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

> This resource contains multiple record sets, fields, and columns for your exploration.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', None)}")
print(f"Dataset Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview

Here, we inspect available **record sets** and their fields. All objects are referenced by their `@id` as per Croissant conventions.

We will print all record set `@id`s in the dataset, their names, and their fields with their respective `@id`s.

In [ ]:
# List available record sets and related field ids
print("Available Record Sets and their fields (by @id):\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"Record set @id: {rs_id}")
    print(f"  Name: {rs_name}")
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f"    Field @id: {field_id} | name: {field_name}")
    print("\n-----------------------------\n")

# Print one or two example records for the first record set
if record_sets:
    first_rs_id = getattr(record_sets[0], '@id', None)
    print(f"Example record(s) from record set @id: {first_rs_id}\n")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(json.dumps(rec, indent=2))
        if i > 0:  # Print up to 2 records
            break

## 3. Data Extraction

We'll extract data from the record set(s) into pandas DataFrames for analysis. We'll reference record set and field ids explicitly via `@id`.

In [ ]:
# Prepare DataFrames for each record set
import collections

dataframes = {}
record_set_ids = [getattr(rs, '@id') for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id}, shape: {df.shape}")

# Print example columns and preview the first record set
if record_set_ids:
    example_id = record_set_ids[0]
    print(f"Columns for record set @id: {example_id}:\n{list(dataframes[example_id].columns)}\n")
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply some common data preprocessing steps, such as filtering, normalization, and grouping. This is all done using specific field `@id`s.

> **Note:** For demonstration, we attempt numeric operations on a sample numeric field (for instance, GAD-7 or PHQ-9 score field).

Edit variables below to match your chosen numeric field and grouping.

In [ ]:
import numpy as np

# Choose your working record set and a numeric field by their @id
record_set_id = record_set_ids[0]  # Update if you want another record set
df = dataframes[record_set_id].copy()

# Try to guess a numeric field (edit if needed for your dataset)
candidate_numeric_fields = [col for col in df.columns if 'score' in col.lower() or df[col].dtype in [np.int64, np.float64]]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    # fallback to a column user can edit
    numeric_field = df.columns[df.dtypes == 'int64'][0] if any(df.dtypes == 'int64') else df.columns[0]

print(f"Using numeric field (by column name, possibly @id): {numeric_field}\n")

# Drop missing values for this analysis
filtered_df = df.dropna(subset=[numeric_field])
# Filter: Show records with high score e.g. >10
threshold = 10
filtered_high = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Records where {numeric_field} > {threshold}: {len(filtered_high)}\n")
display(filtered_high.head())

# Normalize the field
filtered_high[f"{numeric_field}_normalized"] = (filtered_high[numeric_field] - filtered_high[numeric_field].mean()) / (filtered_high[numeric_field].std() + 1e-9)
print(f"Normalized {numeric_field} (first 5 normalized values):")
display(filtered_high[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field
categorical_candidates = [c for c in df.columns if c not in [numeric_field] and df[c].dtype == 'object']
if categorical_candidates:
    group_field = categorical_candidates[0]
    print(f"Grouping by categorical field: {group_field}")
    grouped = filtered_high.groupby(group_field)[numeric_field].mean().reset_index()
    display(grouped.head())
else:
    print("No suitable categorical grouping field found.")

## 5. Visualization

Let's visualize the distribution of our chosen numeric field and show its distribution, optionally grouped by a categorical attribute.

> **Tip:** For richer visualizations, edit variables below to pick different field `@id`s or columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Optional: Boxplot by group (if group_field exists)
if 'group_field' in locals() and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we explored the **FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the `mlcroissant` library. We demonstrated how to:
- Load and interpret the dataset using Croissant's schema and metadata,
- Examine available record sets, their `@id`s, and fields,
- Load records into pandas DataFrames by record set `@id`,
- Perform example exploratory data analysis and normalization/filtering steps on a sample numeric field referenced via `@id`,
- Visualize field distributions and summary statistics.

To perform deeper analysis, consult the dataset documentation and reference specific entity `@id`s as needed. This dataset supports further work in mental health analytics, AI-ready workflows, and open public health research for Africa.